# Clase 147 — Whisper ASR (transcripción + traducción)

Whisper (OpenAI 2022) = encoder-decoder transformer entrenado en 680k h de audio multilingüe.
Pipeline: audio → mel-spectrogram → encoder → decoder autoregressive → texto.
Fallback completo si `whisper` no está instalado.

In [ ]:
USE_WH = False
try:
    import whisper
    USE_WH = True
    print('whisper disponible')
except Exception as e:
    print('whisper no disponible. Fallback: mel-spec desde scratch + API conceptual. Motivo:', type(e).__name__)

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import stft
np.random.seed(42)

## 1. Audio sintético: 440 Hz + 660 Hz (acorde)

In [ ]:
sr = 16000   # Whisper espera 16 kHz
duration = 2.0
t = np.linspace(0, duration, int(sr * duration), endpoint=False)
audio = 0.5 * np.sin(2*np.pi*440*t) + 0.3 * np.sin(2*np.pi*660*t) + 0.1 * np.sin(2*np.pi*880*t)
audio += np.random.normal(0, 0.02, audio.shape)
print(f'audio: {audio.shape}, sr={sr}, dur={duration}s')

plt.figure(figsize=(10, 2)); plt.plot(t[:500], audio[:500]); plt.title('audio (primeros 500 samples)'); plt.show()

## 2. Mel-spectrogram desde scratch (scipy.fft)

In [ ]:
def hz_to_mel(hz): return 2595 * np.log10(1 + hz / 700)
def mel_to_hz(mel): return 700 * (10**(mel / 2595) - 1)

def mel_filterbank(sr, n_fft, n_mels):
    mel_min, mel_max = hz_to_mel(0), hz_to_mel(sr / 2)
    mel_points = np.linspace(mel_min, mel_max, n_mels + 2)
    hz_points = mel_to_hz(mel_points)
    bin_points = np.floor((n_fft + 1) * hz_points / sr).astype(int)
    fb = np.zeros((n_mels, n_fft // 2 + 1))
    for m in range(1, n_mels + 1):
        l, c, r = bin_points[m-1], bin_points[m], bin_points[m+1]
        for k in range(l, c): fb[m-1, k] = (k - l) / max(c - l, 1)
        for k in range(c, r): fb[m-1, k] = (r - k) / max(r - c, 1)
    return fb

n_fft, hop, n_mels = 400, 160, 80
f, t_, Z = stft(audio, fs=sr, nperseg=n_fft, noverlap=n_fft - hop)
mag = np.abs(Z)                                  # (n_fft/2+1, frames)
fb = mel_filterbank(sr, n_fft, n_mels)           # (n_mels, n_fft/2+1)
mel = fb @ mag                                   # (n_mels, frames)
log_mel = np.log10(mel + 1e-10)
print(f'mel-spec shape: {log_mel.shape} (n_mels, frames)')

plt.figure(figsize=(10, 3))
plt.imshow(log_mel, origin='lower', aspect='auto', cmap='magma')
plt.title('log-mel spectrogram (Whisper input format)'); plt.xlabel('frame'); plt.ylabel('mel bin'); plt.colorbar()
plt.show()

## 3. Pipeline conceptual Whisper

```
audio (16kHz mono)
  → log-mel-spec (80 bins, 25ms window, 10ms hop)
  → encoder transformer (conv1d×2 + N encoder blocks)
  → cross-attention
  → decoder autoregressive (con special tokens <|en|><|transcribe|><|notimestamps|>)
  → text tokens (BPE 50k vocab multilingüe)
```

## 4. API conceptual

In [ ]:
if USE_WH:
    # En lab real: model = whisper.load_model('tiny'); result = model.transcribe(audio)
    print('código real:')
    print('  model = whisper.load_model("tiny")        # 39M params, ~75MB')
    print('  result = model.transcribe("audio.mp3")')
    print('  print(result["text"], result["language"])')
else:
    print('Fallback: no se puede transcribir audio sintético sin modelo entrenado.')
    print('API equivalente (HuggingFace):')
    print('  from transformers import pipeline')
    print('  asr = pipeline("automatic-speech-recognition", model="openai/whisper-tiny")')
    print('  asr("audio.mp3")  # → {"text": "..."}')

## 5. Transcribe vs Translate

Whisper soporta dos tareas con un mismo modelo:

```python
# transcribe: mantiene el idioma del audio
result = model.transcribe('spanish.mp3', task='transcribe')
# → "Hola, cómo estás?"  (output en español)

# translate: traduce a INGLÉS siempre
result = model.transcribe('spanish.mp3', task='translate')
# → "Hello, how are you?"
```

El modelo recibe tokens especiales `<|es|><|transcribe|>` o `<|es|><|translate|>` al inicio del prompt del decoder.

Variantes: `tiny` (39M), `base` (74M), `small` (244M), `medium` (769M), `large-v3` (1.55B).
Distil-Whisper: 6x más rápido, ~1% WER peor.
WhisperX: word-level timestamps + speaker diarization.

## 6. Output structure

In [ ]:
fake_result = {
    'text': 'Hello world this is whisper',
    'language': 'en',
    'segments': [
        {'id': 0, 'start': 0.0, 'end': 1.5, 'text': 'Hello world', 'avg_logprob': -0.21},
        {'id': 1, 'start': 1.5, 'end': 3.0, 'text': 'this is whisper', 'avg_logprob': -0.18},
    ],
}
print('language:', fake_result['language'])
print('text:', fake_result['text'])
for s in fake_result['segments']:
    print(f"  [{s['start']:5.2f}→{s['end']:5.2f}] {s['text']:30s} (logprob {s['avg_logprob']:.2f})")

## Conclusiones

- Whisper = ASR multilingüe robust to noise, accents, code-switching.
- Input fijo: 80-mel × 3000 frames (30s windows; >30s se chunk-ea).
- Tasks: transcribe (mismo idioma) o translate (→ inglés).
- Para producción: faster-whisper (CTranslate2) o WhisperX (timestamps + diarization).
- Distil-Whisper para edge / on-device.